# M19 lab solution: End-to-End RAG Systems

**Brief.** Module lab, AI research assistant: package ingestion, retrieval, structured answers with citations and session history into one class (05/10/11).

Run with the default environment: `uv run jupyter lab`. The class follows `mandatory/notebook/lab_ai_research_assistant.ipynb`, with an in-memory FAISS index instead of a persisted Chroma one and citations checked in code.

Open this notebook from `lab/solution/`, so `../data/` resolves to the lab's own data folder.

## Setup

In [ ]:
import json
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # API keys from the .env file at the repo root
sys.path.insert(0, str(Path("../../../../src").resolve()))

from lab_kit import baseline, hit_rate, is_hit, load_handbook, load_jsonl, mrr, recall_at_k  # noqa: E402

DATA = Path("../data/larkspur")
handbook = load_handbook(DATA / "handbook")
golden = load_jsonl(DATA / "golden.jsonl")
by_id = {g["id"]: g for g in golden}
scored = [g for g in golden if g["evidence_spans"]]
print(len(handbook), "handbook files,", len(golden), "golden questions,", len(scored), "with evidence")

In [ ]:
from helpers import get_embeddings, get_experientiallabs_llm

llm = get_experientiallabs_llm()
embeddings = get_embeddings()
LLM_NAME = llm.model_name
EMBEDDING_NAME = getattr(embeddings, "model", None) or getattr(embeddings, "endpoint", None)
print("LLM:", LLM_NAME, "| embeddings:", EMBEDDING_NAME)

## 1. The `ResearchAssistant` class

- **Ingestion**: `add_handbook` splits each file (recursive, 500 characters, 50 overlap) and stores the chunks with `source` and `section`.
- **Retrieval**: a follow-up question is first rewritten into a standalone one from the session history, then searched (top 4).
- **Structured answer**: `ResearchAnswer` holds the answer, a confidence and citations, each a source file and a quote. Every quote is checked against the retrieved chunks in code; a quote that is not in them is marked unverified.
- **Session history**: one `InMemoryChatMessageHistory` per session id, passed to both the rewrite and the answer prompts.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field

from lab_kit import normalise


class Citation(BaseModel):
    source: str = Field(description="File name of the passage, as shown in brackets.")
    quote: str = Field(description="Words copied exactly from that passage.")


class ResearchAnswer(BaseModel):
    answer: str = Field(description="The answer, or 'I don't know' when the passages do not contain it.")
    confidence: str = Field(description="high, medium or low, from how directly the passages answer the question.")
    citations: list[Citation] = Field(description="One citation per passage the answer uses.")


class ResearchAssistant:
    def __init__(self, llm, embeddings, k=4):
        self.llm, self.embeddings, self.k = llm, embeddings, k
        self.splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        self.store = None
        self.sessions: dict[str, InMemoryChatMessageHistory] = {}
        self.rewrite = ChatPromptTemplate.from_messages([
            ("system", "Rewrite the user's last question as one standalone question, using the conversation for context. "
                       "Return only the question."),
            MessagesPlaceholder("history"),
            ("human", "{question}"),
        ]) | llm
        self.answer_chain = ChatPromptTemplate.from_messages([
            ("system", "You answer Larkspur Cloud customers from the handbook passages only. Quote the passages exactly "
                       "in citations. If they do not contain the answer, say I don't know and cite nothing."),
            MessagesPlaceholder("history"),
            ("human", "Passages:\n{context}\n\nQuestion: {question}"),
        ]) | llm.with_structured_output(ResearchAnswer)

    def add_handbook(self, pages):
        docs = [Document(page_content=c, metadata={"source": p["source"], "section": p["title"]})
                for p in pages for c in self.splitter.split_text(p["text"])]
        if self.store is None:
            self.store = FAISS.from_documents(docs, self.embeddings)
        else:
            self.store.add_documents(docs)
        return len(docs)

    def history(self, session_id):
        return self.sessions.setdefault(session_id, InMemoryChatMessageHistory())

    def retrieve(self, question, session_id):
        past = self.history(session_id).messages
        standalone = self.rewrite.invoke({"history": past, "question": question}).content if past else question
        return standalone, self.store.similarity_search(standalone, k=self.k)

    def ask(self, question, session_id="default"):
        standalone, docs = self.retrieve(question, session_id)
        context = "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)
        hist = self.history(session_id)
        result = self.answer_chain.invoke({"history": hist.messages, "context": context, "question": question})
        chunk_text = [normalise(d.page_content) for d in docs]
        verified = [any(normalise(c.quote) in t for t in chunk_text) for c in result.citations]
        hist.add_user_message(question)
        hist.add_ai_message(result.answer)
        return {"question": question, "standalone": standalone, "answer": result, "verified": verified,
                "sources": [d.metadata["source"] for d in docs]}

## 2. Ingest and run a three-turn session

In [ ]:
assistant = ResearchAssistant(llm, embeddings)
print(assistant.add_handbook(handbook), "chunks ingested")

turns = ["How long does a refund take?", "Does that apply to monthly plans too?", "How do I ask for one?"]
session = []
for q in turns:
    out = assistant.ask(q, session_id="customer-42")
    session.append(out)
    print(f"Q: {q}\n   standalone: {out['standalone']}\n   A: {out['answer'].answer}\n   confidence: {out['answer'].confidence}")
    for c, ok in zip(out["answer"].citations, out["verified"]):
        print(f"   [{c.source}] {'verified' if ok else 'NOT in the retrieved chunks'}: {c.quote[:90]}")
    print()

## Results

In [ ]:
import pandas as pd

print("run:", time.strftime("%Y-%m-%d"), "| LLM:", LLM_NAME, "| embeddings:", EMBEDDING_NAME)
print(pd.DataFrame([{"turn": i + 1, "question": s["question"], "standalone": s["standalone"],
                     "citations": len(s["answer"].citations), "verified": sum(s["verified"]),
                     "confidence": s["answer"].confidence} for i, s in enumerate(session)]).to_string(index=False))
print("messages in session history:", len(assistant.history("customer-42").messages))

## What this shows

- Turns 2 and 3 only make sense with turn 1: the standalone rewrite is what lets retrieval find the refund passages again, and the printed rewrite shows whether it did.
- Citations are checked in code against the chunks that were retrieved. A quote the model paraphrased fails that check, which is the failure a highlighter that matches silently would hide.
- The class keeps each concern in one method, so a later module can swap the retriever (hybrid, reranked) without touching history or citations.